# A tiny stable diffusion: from a sentence to a face

> **Warning: do the two exercises first.** This notebook puts two earlier notebooks together, and its code gives away the answers of both of their exercises. If you have not done them yet, do them first:
>
> https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/happy-sad-diffusion/happy_sad_diffusion.ipynb
>
> https://colab.research.google.com/github/eth-bmai-hs26/w4-cx-public/blob/main/text-embeddings/text_embeddings.ipynb

Stable Diffusion draws a picture from a sentence with two networks. A **text encoder**, the text half of CLIP, turns the sentence into numbers. A **denoiser**, the UNet of the slides, turns noise into a picture step by step and reads those numbers at every step. The text encoder is trained first, on its own, and then **frozen**: nothing in it changes any more. The denoiser then learns from pictures paired with **captions**, sentences that describe them.

This notebook builds the same two parts, tiny, out of two earlier notebooks:

* the **text embedder** of "Text embeddings from scratch": a sentence becomes three numbers, how happy, how sad and how surprised it sounds;
* the **denoiser** of the happy and sad extension: noise becomes a face of $9 \times 9$ pixels, happy, sad or surprised, and the denoiser reads three numbers at every step.

They are trained the way Stable Diffusion is: the text embedder first, on the thirty sentences, and then frozen; then the denoiser, on the faces, each with a caption of its expression. At the end you type an expression, a word like happy or a sentence like "a smiling child", and it draws the face.

There is nothing to fill in: run the cells from top to bottom, then try your own sentences at the end.

In [ ]:
#@title Imports { display-mode: "form" }
import torch
from torch import nn
import matplotlib.pyplot as plt

## 1. The faces

The faces of the happy and sad extension: two eyes and a mouth that curves up (happy), curves down (sad) or is wide open (surprised), each one pixel higher or lower and one pixel further left or right, $9$ faces per expression, $27$ in all.

In [ ]:
#@title The 27 faces { display-mode: "form" }
EYES = [(2, 3), (2, 5)]                                          # (row, column) of every ink pixel
HAPPY = [(5, 2), (6, 3), (6, 4), (6, 5), (5, 6)]                 # the mouth curves up
SAD = [(6, 2), (5, 3), (5, 4), (5, 5), (6, 6)]                   # the mouth curves down
SURPRISED = [(5, 3), (5, 4), (5, 5), (6, 3), (6, 5), (7, 3), (7, 4), (7, 5)]   # the mouth is wide open
EXPRESSIONS = ["happy", "sad", "surprised"]

faces, expressions = [], []
for expression, mouth in [(0, HAPPY), (1, SAD), (2, SURPRISED)]:
    for dy in [-1, 0, 1]:                                        # one pixel up, none, one pixel down
        for dx in [-1, 0, 1]:                                    # one pixel left, none, one pixel right
            face = torch.zeros(9, 9)
            for row, col in EYES + mouth:
                face[row + dy, col + dx] = 1.0
            faces.append(face.flatten())                         # the 81 pixels in one row
            expressions.append(expression)
faces = torch.stack(faces)                                       # 27 faces, 81 numbers each
expressions = torch.tensor(expressions)                          # their expressions: 0 happy, 1 sad, 2 surprised
print(faces.shape, expressions.shape)

In [ ]:
#@title Picture of the 27 faces { display-mode: "form" }
def show(ax, picture, title=""):
    ax.imshow(picture.reshape(9, 9), cmap="Greys", vmin=0, vmax=1)
    ax.set(xticks=[], yticks=[], title=title)


fig, axes = plt.subplots(3, 9, figsize=(13, 5))
for ax, face, expression in zip(axes.flat, faces, expressions):
    show(ax, face, EXPRESSIONS[expression])
plt.show()

## 2. The captions

The thirty short descriptions of the extension, ten for each expression. They are the **captions**: when the denoiser learns from a happy face, it gets one of the ten happy sentences with it. The **vocabulary** is the list of their different words, as in the text embeddings notebook.

In [ ]:
HAPPY_TEXTS = ["a happy face", "a smiling face", "someone who is glad", "a cheerful person", "she is delighted",
               "he is laughing", "a big smile", "a pleased child", "they look very happy", "a grinning man"]
SAD_TEXTS = ["a sad face", "an unhappy face", "someone who is crying", "a gloomy person", "she is upset",
             "he is miserable", "tears running down a face", "a heartbroken child", "they look very sad",
             "a tearful woman"]
SURPRISED_TEXTS = ["a surprised face", "an astonished face", "someone who is amazed", "a shocked person",
                   "she is stunned", "he is startled", "a mouth wide open in surprise", "a speechless child",
                   "they look very surprised", "an open-mouthed man"]

sentences = HAPPY_TEXTS + SAD_TEXTS + SURPRISED_TEXTS            # 30 captions: 0 to 9 happy, 10 to 19 sad, 20 to 29 surprised
labels = torch.tensor([0] * 10 + [1] * 10 + [2] * 10)            # their expressions: 0 happy, 1 sad, 2 surprised
vocabulary = sorted(set(" ".join(sentences).split()))            # every different word, once
print(len(sentences), "captions,", len(vocabulary), "words:", vocabulary)

## 3. The text embedder

The model of the text embeddings notebook. Every word has two numbers of its own in a table, `three_numbers` averages the rows of a sentence's words, and a wide network turns the average into three numbers, trained towards the one-hot of the sentence's expression: $[1, 0, 0]$ for happy, $[0, 1, 0]$ for sad, $[0, 0, 1]$ for surprised.

`prompt_from(sentence)` is what the denoiser will read, the **prompt**: the three numbers of the sentence, each cut to between $0$ and $1$. The cut matters for a single word: "happy" alone has no other word to average with, so its numbers come out larger than any caption's, about $2$ for happy, and the denoiser has never seen such a prompt. Words that are not in the vocabulary are left out, and a sentence with no word the embedder knows gets the empty prompt $[0, 0, 0]$.

In [ ]:
torch.manual_seed(0)   # seeded: the same starting numbers on every computer

embedding = nn.Embedding(len(vocabulary), 2)                     # a table: 2 numbers for every word
nn.init.normal_(embedding.weight, std=0.02)                      # every word starts close to (0, 0)

network = nn.Sequential(
    nn.Linear(2, 100), nn.ReLU(),                                # wide: one hidden layer of 100 neurons
    nn.Linear(100, 3),                                           # out: the 3 numbers of the sentence
)


def three_numbers(sentence):
    ids = torch.tensor([vocabulary.index(word) for word in sentence.split()])   # the number of every word
    vectors = embedding(ids)                                     # its row of the table: 2 numbers per word
    average = vectors.mean(dim=0)                                # their average: 2 numbers for the sentence
    return network(average)                                      # 3 numbers for the sentence


def prompt_from(sentence):
    known = [word for word in sentence.lower().split() if word in vocabulary]   # words it never saw are left out
    if not known:
        return torch.zeros(3)                                    # no word it knows: the empty prompt
    with torch.no_grad():                                        # the text embedder is only read here
        return three_numbers(" ".join(known)).clamp(0, 1)        # each number cut to between 0 and 1

## 4. Training the text embedder, then freezing it

The loop of the text embeddings notebook: the average squared difference between the three numbers of the thirty sentences and their one-hots, until it is below $0.01$. It takes a few dozen steps.

Then the text embedder reads the thirty captions once, and from here on it is **frozen**: the denoiser's training reads these thirty prompts and never changes the table or the wide network, as Stable Diffusion's denoiser reads CLIP.

In [ ]:
targets = torch.eye(3)[labels]                                   # the one-hot of every sentence's expression
optimizer = torch.optim.Adam(list(embedding.parameters()) + list(network.parameters()), lr=0.01)

loss = 1.0                                                       # any start above the threshold
while loss > 0.01:
    numbers = torch.stack([three_numbers(sentence) for sentence in sentences])   # 30 sentences, 3 numbers each
    loss = ((numbers - targets) ** 2).mean()                     # the average squared difference
    optimizer.zero_grad()                                        # forget the gradients of the previous step
    loss.backward()                                              # compute the gradient of the loss
    optimizer.step()                                             # move every weight, and every word, a small step
    print(f"\rloss: {loss.item():.4f}", end="")

In [ ]:
caption_prompts = torch.stack([prompt_from(sentence) for sentence in sentences])   # 30 prompts, 3 numbers each
for i in [0, 10, 20]:
    print(f"{sentences[i]!r:20} prompt {caption_prompts[i].numpy().round(2)}")

## 5. The denoiser

The denoiser of the happy and sad extension, unchanged. The forward process fades a face into one noise picture over $T = 20$ steps. The network reads $81 + 1 + 3 = 85$ numbers, the noisy picture, the step $t$ (as $t/T$) and the prompt, and guesses the clean face; `previous_picture` moves the picture a $1/t$ part of the way towards that guess, which is its estimate of the picture one step earlier.

In [ ]:
T = 20                                                           # the number of steps from a face to pure noise


def add_noise(face, noise, t):
    return (1 - t / T) * face + (t / T) * noise                 # the picture at step t


torch.manual_seed(0)   # seeded again: the denoiser starts the same whatever the text embedder did
net = nn.Sequential(
    nn.Linear(81 + 1 + 3, 128), nn.ReLU(),                       # in: the noisy picture, the step and the prompt
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 81),                                          # out: its guess of the clean face
)


def previous_picture(x, t, prompt):
    guess = net(torch.cat([x, t / T, prompt], dim=1))            # the network's guess of the clean face
    return x + (guess - x) / t                                   # one step from x_t towards it: the estimate of x_{t-1}

## 6. Training the denoiser on faces with their captions

The loop of the happy and sad notebook, with one change: the prompt of each face is a **caption**. Every step picks $256$ faces at random and, for each face, one of the ten captions of its expression, also at random; the face's prompt is that caption's three numbers, read from the frozen text embedder. Only the denoiser learns here. The loop stops at the threshold of the happy and sad notebook, after a few thousand steps.

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)        # only the denoiser's weights: the text embedder is frozen

loss = 1.0                                                       # any start above the threshold
while loss > 0.012:
    pick = torch.randint(0, 27, (256,))                          # 256 faces picked at random
    face = faces[pick]
    caption = 10 * expressions[pick] + torch.randint(0, 10, (256,))   # one of the 10 captions of each face's expression
    prompt = caption_prompts[caption]                            # its 3 numbers, from the frozen text embedder
    noise = torch.randn_like(face)                               # a noise picture for each face
    t = torch.randint(1, T + 1, (256, 1)).float()                # a step between 1 and T for each face
    x_t = add_noise(face, noise, t)                              # the noisy picture at step t
    x_previous = add_noise(face, noise, t - 1)                   # the true picture one step earlier
    loss = ((previous_picture(x_t, t, prompt) - x_previous) ** 2).sum(dim=1).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"\rloss: {loss.item():.4f}", end="")

## 7. From a sentence to a face

The three steps of the slides, with one step in front of them: the frozen text embedder turns the sentence into its prompt. The denoiser then reads that prompt at every step, from noise down to the face.

In [ ]:
def faces_from(sentence, how_many=8):
    prompt = prompt_from(sentence)                               # the frozen text embedder reads the sentence
    with torch.no_grad():
        x = torch.randn(how_many, 81)                            # 1) sample noise: x_T
        prompts = prompt.repeat(how_many, 1)                     # the same prompt for every face
        t = T
        while t > 0:                                             # 2) for t = T, T-1, ..., 1:
            step = torch.full((how_many, 1), float(t))
            x = previous_picture(x, step, prompts)               #    estimate x_{t-1} from x_t
            t = t - 1
    return x, prompt                                             # 3) output x_0, with the prompt it read

Eight faces for each of six prompts: the three expressions as single words, then three sentences that are not among the captions, made of words the text embedder knows.

In [ ]:
#@title Faces for six prompts { display-mode: "form" }
examples = ["happy", "sad", "surprised", "a smiling child", "he is upset", "she is amazed"]
fig, axes = plt.subplots(len(examples), 8, figsize=(12, 10))
for row, sentence in zip(axes, examples):
    pictures, prompt = faces_from(sentence)
    for ax, picture in zip(row, pictures):
        show(ax, picture)
    row[0].set_title(f'"{sentence}": prompt {[round(v, 2) for v in prompt.tolist()]}', loc="left")
plt.show()

## 8. Try it

* Write your own sentence in the cell below and run it. The cell prints the prompt and the words the text embedder never saw.
* What prompt does "a smiling and crying face" get, and which faces does the denoiser draw from it?
* "tears of joy" and "a face that is not happy": which words does the text embedder read, and which face comes out?
* Compare "a smiling man" and "a smiling woman". The only caption with the word woman is "a tearful woman": what does that do to the faces?
* "good morning" has no word the text embedder knows, so the denoiser gets the empty prompt $[0, 0, 0]$, which it never saw in training. What does it draw?

In [ ]:
sentence = "a cheerful child"                                    # write your own sentence here
pictures, prompt = faces_from(sentence, how_many=10)
print("prompt:", prompt.numpy().round(2), "  never saw:", [w for w in sentence.lower().split() if w not in vocabulary])
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
for ax, picture in zip(axes, pictures):
    show(ax, picture)
plt.show()

## 9. What Stable Diffusion does differently

The same two parts and the same training, at a much larger scale.

* Its text encoder is the text half of CLIP, a transformer trained on $400$ million pictures with their captions. It gives $768$ numbers for each of up to $77$ tokens of the prompt, where ours gives three numbers for the whole sentence.
* Its denoiser, a UNet, reads those numbers through cross-attention, the attention of weekend 2, at every step; ours glues the three numbers to the noisy picture.
* It draws in the code of an autoencoder, a variational one as this morning: $64 \times 64 \times 4$ numbers for a picture of $512 \times 512$ pixels, which the decoder turns into the picture at the end. Ours draws the $81$ pixels themselves.
* Its denoiser guesses the noise, as on the slides; ours guesses the clean face, as in the happy and sad notebook.